# Ingestión del archivo `language_role.json`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer el archivo JSON usando `DataFrameReader` de Spark

In [0]:
language_role_schema = "roleId INT, languageRole STRING"

language_role_df = (spark.read 
    .schema(language_role_schema)
    .option("multiLine", True)
    .json(f"{bronze_folder_path}/{v_file_date}/language_role.json")
)
display(language_role_df)

roleId,languageRole
1,Original
2,Spoken


## 2. Cambiar el nombre de las columnas según lo requerido

In [0]:
language_role_renamed_df = (language_role_df
    .withColumnRenamed("roleId", "role_id")
    .withColumnRenamed("languageRole", "language_role")
)

## 4. Agregar las columnas `ingestion_date` y `environmate` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

language_role_renamed_final_df = add_ingestion_date(language_role_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))


## 5. Escribir datos en el datalake en formato `Parquet`

In [0]:
language_role_renamed_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages_roles")

In [0]:
%sql
SELECT * FROM movie_silver.languages_roles

role_id,language_role,ingestion_date,enviroment,file_date
1,Original,2026-09-13T17:52:17.612Z,,2024-12-16
2,Spoken,2026-09-13T17:52:17.612Z,,2024-12-16
